<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Module 5 &middot; Make it better</div><div style="font-size:15px;color:#cfd4ff;">The hill climb &middot; about 14 minutes</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#4de1ff;color:#07062e;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 5. Make it better (v1 to v2)

**Time:** about 14 minutes.

**The question we're starting with:** how do I know my agent really got *better*, and not just *different*?

In Module 4 we found problems by reading. That doesn't scale, and "it feels better" isn't proof. This time we'll do two things that do:

1. **Define "good" as a number.** We'll build a **rubric evaluator** from our own traces and gap list, then score our agents with it.
2. **Let the agent optimizer climb for us.** It rewrites the instructions, tries several candidates, and scores each with *our* rubric.

The key idea: **the optimizer is only as trustworthy as the rubric guiding it.** A bad ruler makes a confident, wrong climb.

| Our name | Foundry version | What changed |
|---|---|---|
| v0 | 1 | Generic instructions |
| v1 | 2 | Better instructions (Module 4) |
| v2 | 3 or later | **Today:** the optimizer's best candidate, promoted |


---

## 5.1 Which versions do I have to compare?

We'll need the agent's saved versions, so let's list them and give them our lab names.

In [ ]:
import sys
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
import lab

project_client, openai_client = lab.connect()
dev_cases, test_cases = lab.load_cases("dev"), lab.load_cases("test")

# Foundry numbers versions from 1. Oldest first: v0, v1, and anything newer.
versions = sorted(project_client.agents.list_versions(lab.AGENT_NAME), key=lambda v: int(v.version))
v0, v1 = versions[0], versions[1]
print(f"v0 = Foundry version {v0.version}   v1 = Foundry version {v1.version}")
print(f"{len(dev_cases)} dev questions, {len(test_cases)} test questions")

## 5.2 How do I turn "good" into a number?

A **rubric evaluator** is an AI judge with a checklist. It reads a question and an answer and gives a score from 1 to 5.

We could write the checklist ourselves, and Foundry can also **draft one from our agent's real traces**. We'll give it two ingredients:

- the traces from Module 4 (what the agent really did), and
- the gap list we wrote (what we *think* matters).

**Before you run it:** paste your gaps from Module 4 below. The better they are, the better the rubric.

In [ ]:
# Paste the gaps we wrote in Module 4 here, in our own words.
our_gaps = [
    "Replace me with a gap we found",
    "Replace me with another gap",
]

from azure.ai.projects.models import (
    EvaluatorGenerationInputs,
    EvaluatorGenerationJob,
    PromptEvaluatorGenerationJobSource,
    TracesEvaluatorGenerationJobSource,
)

job = EvaluatorGenerationJob(
    inputs=EvaluatorGenerationInputs(
        sources=[
            # Real conversations from the last day, so the rubric is grounded in what happened.
            TracesEvaluatorGenerationJobSource(
                agent_name=lab.AGENT_NAME,
                start_time=datetime.now(timezone.utc) - timedelta(hours=24),
                description="Traces from our TrailMate practice questions",
            ),
            # Our own words about what matters, so the judge looks for the right things.
            PromptEvaluatorGenerationJobSource(
                prompt="Problems we found with this agent: " + "; ".join(our_gaps),
                description="Our gap list from Module 4",
            ),
        ],
        model=lab.FRONTIER_MODEL,  # the model that writes the rubric
        evaluator_name=f"trailmate-rubric-{uuid.uuid4().hex[:6]}",
        evaluator_display_name="Our TrailMate rubric",
    )
)

poller = project_client.beta.evaluators.begin_create_generation_job(job=job)
rubric = poller.result()
print("Rubric created:", rubric.name, "version", rubric.version)

## 5.3 Can I trust the rubric I just generated?

Never use a ruler you haven't looked at. Let's read what Foundry wrote, then read **our reference rubric** side by side.

In [ ]:
# What Foundry drafted from our traces and gaps.
print(rubric)

And here is our reference rubric, written by hand, to compare against.

In [ ]:
# Our reference rubric, written by hand for comparison.
print((lab.DATA / "evaluators" / "trailmate-quality.yaml").read_text(encoding="utf-8"))

**Think:**
- What does the generated rubric check that ours doesn't, and the other way round?
- Does it catch "water-resistant" being upgraded to "waterproof"?
- Does it punish an agent for answering something it should have declined?

If the rubric misses something important, our scores will be blind to it. That's the part worth slowing down for.

## 5.4 How do I score every version the same way?

Foundry needs our questions as **datasets**. We upload `dev` (used to tune and score) and `test` (the held-out set the optimizer *never* sees). Then we create an **evaluation** that ties our rubric to a dataset format.

In [ ]:
from azure.core.exceptions import ResourceExistsError


def upload(split):
    """Upload one question set as a versioned dataset, reusing it if it's already there."""
    name = f"trailmate-{split}"
    try:
        return project_client.datasets.upload_file(name=name, version="1", file_path=str(lab.DATA / f"{split}.jsonl"))
    except ResourceExistsError:
        return project_client.datasets.get(name=name, version="1")


dev_dataset, test_dataset = upload("dev"), upload("test")
print("Datasets:", dev_dataset.name, "and", test_dataset.name)

Next, the evaluation itself. It ties our rubric to the shape of our questions, and we'll reuse it for every version so every score comes from the same ruler.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

# One evaluation = our questions' shape + our rubric. We reuse it for every version,
# so every score comes from the same ruler.
evaluation = openai_client.evals.create(
    name="TrailMate rubric evaluation",
    data_source_config=DataSourceConfigCustom(
        type="custom",
        item_schema={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        include_sample_schema=True,
    ),
    testing_criteria=[
        TestingCriterionAzureAIEvaluator(
            type="azure_ai_evaluator",
            name="Our rubric",
            evaluator_name=rubric.name,
            initialization_parameters={"deployment_name": lab.FRONTIER_MODEL},
            # The judge sees the question and the agent's final answer.
            data_mapping={"query": "{{item.query}}", "response": "{{sample.output_text}}"},
        )
    ],
)
print("Evaluation ready:", evaluation.id)

# Remember these so the next notebook can pick up where we left off.
lab.save_state(
    rubric=rubric.name,
    evaluation_id=evaluation.id,
    dev_dataset_id=dev_dataset.id,
    test_dataset_id=test_dataset.id,
)

## 5.5 How high are we right now?

Now we score v0 and v1 on the **dev** set. Each is run **twice** and averaged, because a judge can wobble a little between runs. The gap between the two runs (the **spread**) tells us how much noise to expect, so we know how big a gain has to be before we believe it.

**Predict:** how many points will v1 beat v0 by?

In [ ]:
baseline = [
    lab.run_repeated_eval(openai_client, evaluation.id, dev_dataset.id, lab.AGENT_NAME, v.version, label)
    for v, label in [(v0, "v0 on dev"), (v1, "v1 on dev")]
]

scoreboard = pd.DataFrame(
    [{"version": r["label"], "score": r["score"], "spread": r["spread"], "tokens": r["tokens"], "errored": r["errored"]}
     for r in baseline]
)
scoreboard

**Expected:** two rows (v0 and v1), each with a score, a spread, a token count and `errored` at 0.

**Read it carefully.**
- Did v1 beat v0? By more than the spread? A gain smaller than the spread could just be noise.
- Any `errored` items? Those weren't scored, so that version was graded on fewer questions.

Now the useful view: **which of our six ways are weakest?**

In [ ]:
by_way = pd.DataFrame({r["label"]: lab.scores_by_way(r["per_item"], dev_cases) for r in baseline})
by_way

**Think:** which row is lowest? That's where the optimizer has the most room to help, and where a good rewrite should show up.

## 5.6 Can the optimizer climb for me?

The **agent optimizer** does what you'd do by hand, only faster and wider. It rewrites the instructions, tries several **candidates**, scores each on a set of questions with our rubric, and reports back.

Three choices are ours:

- **Training set:** the questions it learns from. We give it `dev`. It never sees `test`.
- **Judge:** our rubric.
- **Budget:** how many candidates to try. More tries cost more and may climb higher.

**Go deeper:** the optimizer proposes candidate rewrites, scores each one on the training questions with our rubric, and reports the best. `max_candidates` caps how many get fully evaluated, which is the main lever on cost.

First, a sensible habit: **ask what it will cost before we spend it.**

In [ ]:
from azure.ai.projects.models import (
    AgentOptimizationCandidateSearchConfiguration,
    AgentOptimizationConfiguration,
    AgentOptimizationEstimateInputs,
    AgentOptimizationEvaluationConfiguration,
    AgentOptimizationEvaluator,
    AgentOptimizationFoundryAgentTargetConfiguration,
    AgentOptimizationJob,
    AgentOptimizationModelConfiguration,
    AgentOptimizationSpace,
    AgentOptimizationTargetCompletionDatasetReferenceDataSource,
    AgentOptimizationTargetCompletionEvaluationSet,
    EvaluationModelConfiguration,
)

target = AgentOptimizationFoundryAgentTargetConfiguration(name=lab.AGENT_NAME, version=str(v1.version))
writer = AgentOptimizationModelConfiguration(model=lab.FRONTIER_MODEL)  # the model that rewrites instructions

settings = AgentOptimizationConfiguration(
    evaluation_configuration=AgentOptimizationEvaluationConfiguration(
        training_set=AgentOptimizationTargetCompletionEvaluationSet(
            source=AgentOptimizationTargetCompletionDatasetReferenceDataSource(name=dev_dataset.name, version="1")
        ),
        evaluators=[AgentOptimizationEvaluator(name=rubric.name)],  # our rubric is the judge
        evaluation_model=EvaluationModelConfiguration(model=lab.FRONTIER_MODEL),
    ),
    candidate_search_configuration=AgentOptimizationCandidateSearchConfiguration(max_candidates=3),
    agent_optimization_space=AgentOptimizationSpace(target_attributes=["instructions"]),  # one lever: instructions
)

estimate = project_client.beta.agents.estimate_optimization_job(
    AgentOptimizationEstimateInputs(
        optimization_model_configuration=writer, optimization_configuration=settings, target_configuration=target
    )
)
print(estimate)

**Expected:** an estimate with call counts and a cost range.

**Think:** is that cost sensible for what we expect to gain? Notice we limited it to **three candidates** and **one lever** (instructions). We're changing one thing at a time, even though the optimizer could do more.

Ready? Start the job and wait for it. This is the longest wait in the lab, so now is a good moment to think about the next section.

In [ ]:
poller = project_client.beta.agents.begin_create_optimization_job(
    AgentOptimizationJob(
        target_configuration=target,
        optimization_model_configuration=writer,
        optimization_configuration=settings,
        display_name="TrailMate: improve instructions",
    )
)
job_id = poller.details["job_id"]
print("Optimization job started:", job_id)

result = poller.result()  # waits until the job finishes
print("Finished. Best score:", result.candidate_summary.best_score, "| baseline:", result.candidate_summary.baseline_score)

## 5.7 What did the optimizer actually change?

Never promote blindly. A candidate is a *proposal*. Let's see each one, its score, and **what it actually changed**.

In [ ]:
candidates = list(project_client.beta.agents.list_optimization_candidates(job_id, expand=["mutations"]))

rows = []
for c in candidates:
    new_text = next((m.value for m in (c.output.mutations if c.output else []) if m.type == "instructions"), None)
    rows.append({
        "candidate": c.name,
        "status": c.status,
        "score": c.evaluation.score if c.evaluation else None,
        "avg_tokens": c.evaluation.avg_tokens if c.evaluation else None,
        "rationale": (c.rationale or "")[:90],
        "instructions": new_text,
    })

pd.DataFrame(rows).drop(columns=["instructions"])

Scores only tell us who won. To know *why*, let's read the winner's instructions in full.

In [ ]:
# Read the winning instructions in full. Do they make sense to a human?
best = max((r for r in rows if r["score"] is not None and r["instructions"]), key=lambda r: r["score"])
print("Best candidate:", best["candidate"], "| score:", best["score"])
print("\n" + best["instructions"])

**Think:**
- Do these instructions tell the agent what to do when the manual is silent? When a question is off-topic?
- Could a candidate score well by **refusing too much**? That's the trap. A cautious agent never invents anything, and it's also useless.

We'll check for that trap on purpose, in a moment.

## 5.8 How do I keep the winner?

Promotion saves the candidate's instructions as a **new agent version**. Nothing is overwritten.

In [ ]:
best_candidate = next(c for c in candidates if c.name == best["candidate"])
promoted = project_client.beta.agents.promote_optimization_candidate(job_id, best_candidate.candidate_id)
print(promoted.promotion)

# The newest version is our v2.
versions = sorted(project_client.agents.list_versions(lab.AGENT_NAME), key=lambda v: int(v.version))
v2 = versions[-1]
print(f"v2 = Foundry version {v2.version}")

## 5.9 Did it get better, or just more cautious?

Our dev set holds a few **guard cases**: questions the manuals *do* answer, where the right move is to just answer. If v2 learned to hedge or refuse everything, it will fail these.

A quick look with our own eyes, before any scoring:

In [ ]:
hedges = ["not specified", "isn't specified", "doesn't say", "does not say", "don't have", "cannot", "can't"]

for case in [c for c in dev_cases if c["guard"]]:
    answer = lab.ask_agent(openai_client, case["query"], agent_version=v2.version).output_text.strip()
    hedged = any(h in answer.lower() for h in hedges)
    print(f"\n[{case['id']}] {case['query']}")
    print("v2 says:", answer[:300])
    print("Hedged or refused?", "YES, that's a problem" if hedged else "no, it answered")

**Think:** if v2 hedged on a question the manual answers, the optimizer over-corrected. What would we do? (Promote a different candidate, or tighten the rubric so it punishes needless refusals.) If it answered cleanly, great. We just *proved* it, which is better than hoping.

## 5.10 Is the improvement real?

All the scoring so far used `dev`, which the optimizer trained on. A fair test needs questions it **never saw**. That's our held-out `test` set. We score v0, v1 and v2 side by side.

In [ ]:
final = [
    lab.run_repeated_eval(openai_client, evaluation.id, test_dataset.id, lab.AGENT_NAME, v.version, label)
    for v, label in [(v0, "v0 on test"), (v1, "v1 on test"), (v2, "v2 on test")]
]

board = pd.DataFrame(
    [{"version": r["label"], "score": r["score"], "spread": r["spread"], "tokens": r["tokens"], "errored": r["errored"]}
     for r in final]
)
board

That's the scoreboard. But is the gain *real*? We compare it to the noise we measured.

In [ ]:
# Is the gain real? A gain bigger than the noise (spread) is worth believing.
gain = round(final[2]["score"] - final[1]["score"], 2)
noise = max(r["spread"] or 0 for r in final)
print(f"v2 beat v1 by {gain} points. Run-to-run noise was about {noise}.")
print("That looks real." if gain > noise else "That's within the noise. We shouldn't claim a win yet.")

print(pd.DataFrame({r["label"]: lab.scores_by_way(r["per_item"], test_cases) for r in final}))

# Save our test scores so Module 6 can build the full v0 to v3 picture.
lab.save_state(
    v2_version=v2.version,
    test_scores={r["label"]: {k: r[k] for k in ("score", "spread", "tokens", "per_item")} for r in final},
)

**Expected:** one column per version (v0, v1, v2) and one row per way.

**Read the table by row.** Which of the six ways did v2 improve most? Which barely moved? Whatever barely moved is a different kind of problem. An instruction rewrite may not fix it, and that's a clue for Module 6.

## 5.11 Why do I need questions the optimizer never saw? (the aha)

One last experiment. We'll measure the *same* agent (v2) on the questions the optimizer trained on, and compare that gain with the gain on questions it never saw.

In [ ]:
# Same rubric, same agent. Only the questions differ: ones it trained on, and ones it never saw.
v2_dev = lab.run_repeated_eval(openai_client, evaluation.id, dev_dataset.id, lab.AGENT_NAME, v2.version, "v2 on dev")
dev_gain = round(v2_dev["score"] - baseline[1]["score"], 2)

print(f"v1 to v2 gain on dev (what the optimizer trained on): {dev_gain}")
print(f"v1 to v2 gain on test (questions it never saw):       {gain}")

> **Aha:** The gain on questions the optimizer trained on is the gain it *wanted*. The gain on questions it never saw is the gain we can *trust*. If the second number is smaller, that's normal, and it's exactly why a held-out set isn't optional. "Better" only counts when it survives data the climber couldn't see.

We also took away:

- **A rubric turns opinion into a number.** Without it, the optimizer has no direction.
- **The optimizer is a proposer, not a judge.** We read its candidates and checked for over-correction before trusting it.
- **Held-out data keeps us honest.** We tuned on `dev` and proved it on `test`.
- **Noise is real.** A gain smaller than the spread isn't a win yet.
- A good climb is **measured, inspected and then promoted**, in that order.

**Break, fix, learn:** an optimizer can win by being timid. Guard cases catch that.

**Go further (optional):** raise `max_candidates` to 5 and run the optimizer again. Check the estimate first. Does the bigger search find something better, or just spend more?

**Next:** `06-make-it-optimal.ipynb`. Quality is up, but we're still paying a frontier model for every question. Can the router cut the bill and hold the score?